# PlantSeg lesion segmentation - resumable Google Colab training

This notebook trains a practical binary lesion-segmentation model on the official PlantSeg train/validation/test splits. PlantSeg annotations identify diseased regions; they do **not** provide the 18-way disease classifier used elsewhere in this repository. The output is therefore a lesion mask model that can be used as an app preprocessing/visual-explanation component.

The notebook is designed for a free Colab GPU:
- the 1.06 GB Zenodo archive, checkpoints, and exports live in Google Drive;
- `wget -c` resumes an interrupted download;
- MobileNetV3-DeepLabV3 is the default; set `BACKBONE = 'resnet50'` for a stronger but heavier ResNet-50 variant;
- 256 px inputs, AMP, and small batches reduce VRAM use;
- a checkpoint is written during training every `CHECKPOINT_EVERY_STEPS` batches and at every epoch;
- rerunning the setup and training cells automatically resumes from `latest.pt`.

**License note:** PlantSeg is CC BY-NC 4.0 according to the Zenodo record. Keep the downloaded images and resulting weights private unless your intended distribution complies with that license, and retain attribution to the authors and source record.

In [ ]:
# Colab setup. Re-run safely after a runtime disconnect.
%pip install -q opencv-python-headless tqdm scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, os, random, shutil, subprocess, time, zipfile

WORK_ROOT = Path('/content/drive/MyDrive/plantseg_colab')
ARCHIVE = WORK_ROOT / 'plantseg.zip'
EXTRACT_ROOT = WORK_ROOT / 'dataset'
BACKBONE = 'mobilenet_v3'  # alternatives: 'resnet50'
if BACKBONE not in {'mobilenet_v3', 'resnet50'}:
    raise ValueError(f'Unsupported BACKBONE: {BACKBONE}')
ARCHITECTURE = f'deeplabv3_{BACKBONE}'
RUN_ROOT = WORK_ROOT / 'runs' / f'{BACKBONE}_deeplabv3_binary'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

ZENODO_URL = 'https://zenodo.org/api/records/17719108/files/plantseg.zip/content'
EXPECTED_MD5 = '9358a66dff88cdd15c4fe009763c40a3'

# Free Colab-friendly defaults. Increase MAX_EPOCHS across multiple sessions if needed.
IMAGE_SIZE = 256
BATCH_SIZE = 4
NUM_WORKERS = 2
MAX_EPOCHS = 12
MAX_MINUTES_PER_SESSION = 45
CHECKPOINT_EVERY_STEPS = 200
SEED = 42

print('Artifacts:', RUN_ROOT)
print('GPU will be used:', __import__('torch').cuda.is_available())

## 1. Download and locate the official data

The archive is large, so this cell deliberately keeps it in Drive. If Colab disconnects while downloading, run it again; `wget -c` continues the partial file. The checksum is checked only after the download completes.

In [ ]:
EXPECTED_BYTES = 1_057_281_724
if not ARCHIVE.exists() or ARCHIVE.stat().st_size != EXPECTED_BYTES:
    subprocess.run([
        'wget', '-c', '--show-progress', '-O', str(ARCHIVE), ZENODO_URL
    ], check=True)

md5 = hashlib.md5()
with ARCHIVE.open('rb') as f:
    for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
        md5.update(block)
if md5.hexdigest() != EXPECTED_MD5:
    raise RuntimeError(f'Archive checksum mismatch: {md5.hexdigest()}')
print(f'Verified {ARCHIVE} ({ARCHIVE.stat().st_size / 1e9:.2f} GB)')

# Extraction is idempotent. The marker prevents a repeated unzip after a clean run.
marker = EXTRACT_ROOT / '.extracted'
if not marker.exists():
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', '-o', str(ARCHIVE), '-d', str(EXTRACT_ROOT)], check=True)
    marker.touch()

annotation_files = sorted(EXTRACT_ROOT.rglob('annotation_train.json'))
if not annotation_files:
    raise FileNotFoundError('Could not find annotation_train.json after extraction')
DATA_ROOT = annotation_files[0].parent
print('Dataset root:', DATA_ROOT)
print('Annotations:', sorted(p.name for p in DATA_ROOT.glob('annotation_*.json')))

In [ ]:
import cv2
import numpy as np
from PIL import Image

def build_image_index(root):
    index = {}
    for p in root.rglob('*'):
        if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
            index.setdefault(p.name, p)
    return index

IMAGE_INDEX = build_image_index(DATA_ROOT)
if len(IMAGE_INDEX) < 100:
    raise RuntimeError(f'Only found {len(IMAGE_INDEX)} images under {DATA_ROOT}')
print('Indexed images:', len(IMAGE_INDEX))

def load_split(split):
    with (DATA_ROOT / f'annotation_{split}.json').open() as f:
        coco = json.load(f)
    annotations = {}
    for ann in coco['annotations']:
        annotations.setdefault(ann['image_id'], []).append(ann)
    rows = []
    missing = []
    for image in coco['images']:
        path = IMAGE_INDEX.get(image['file_name'])
        if path is None:
            missing.append(image['file_name'])
            continue
        rows.append((path, image, annotations.get(image['id'], [])))
    if missing:
        raise FileNotFoundError(f'{len(missing)} images missing, first: {missing[:3]}')
    return rows

SPLITS = {name: load_split(name) for name in ('train', 'val', 'test')}
print({name: len(rows) for name, rows in SPLITS.items()})

## 2. Dataset and model

    Polygon annotations are rasterized on the fly, so no second copy of the masks is needed. The model predicts background vs. diseased region. Augmentation is applied only to training images and masks together.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large, DeepLabV3_MobileNet_V3_Large_Weights
import torchvision.transforms.functional as TF

class PlantSegDataset(Dataset):
    def __init__(self, rows, train=False):
        self.rows, self.train = rows, train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        path, image_info, anns = self.rows[index]
        image = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if image is None:
            raise OSError(f'Unable to read {path}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        height, width = image.shape[:2]
        mask = np.zeros((height, width), dtype=np.uint8)
        for ann in anns:
            for polygon in ann.get('segmentation', []):
                points = np.asarray(polygon, dtype=np.float32).reshape(-1, 2)
                if len(points) >= 3:
                    cv2.fillPoly(mask, [np.round(points).astype(np.int32)], 1)

        image = Image.fromarray(image)
        mask = Image.fromarray(mask * 255)
        image = TF.resize(image, [IMAGE_SIZE, IMAGE_SIZE], antialias=True)
        mask = TF.resize(mask, [IMAGE_SIZE, IMAGE_SIZE], interpolation=TF.InterpolationMode.NEAREST)
        if self.train:
            if random.random() < 0.5:
                image, mask = TF.hflip(image), TF.hflip(mask)
            if random.random() < 0.3:
                image, mask = TF.vflip(image), TF.vflip(mask)
            if random.random() < 0.3:
                image = TF.adjust_brightness(image, random.uniform(0.8, 1.2))
                image = TF.adjust_contrast(image, random.uniform(0.8, 1.2))
        image = TF.to_tensor(image)
        image = TF.normalize(image, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        mask = torch.from_numpy(np.asarray(mask) > 127).long()
        return image, mask

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

generator = torch.Generator().manual_seed(SEED)
loaders = {
    'train': DataLoader(PlantSegDataset(SPLITS['train'], True), batch_size=BATCH_SIZE,
                        shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
                        persistent_workers=NUM_WORKERS > 0, worker_init_fn=seed_worker,
                        generator=generator, drop_last=True),
    'val': DataLoader(PlantSegDataset(SPLITS['val']), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
                        persistent_workers=NUM_WORKERS > 0),
    'test': DataLoader(PlantSegDataset(SPLITS['test']), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
                        persistent_workers=NUM_WORKERS > 0),
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
weights = DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT
model = deeplabv3_mobilenet_v3_large(weights=weights)
model.classifier[4] = nn.Conv2d(256, 2, kernel_size=1)
model.aux_classifier[4] = nn.Conv2d(10, 2, kernel_size=1)
model.to(device)
print(device, f'{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters')

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
criterion = nn.CrossEntropyLoss()

def dice_loss(logits, target):
    probability = logits.softmax(1)[:, 1]
    target_float = (target == 1).float()
    intersection = (probability * target_float).sum((1, 2))
    return 1 - ((2 * intersection + 1) / (probability.sum((1, 2)) + target_float.sum((1, 2)) + 1)).mean()

def scores(logits, target):
    pred = logits.argmax(1)
    truth = target == 1
    positive = pred == 1
    intersection = (positive & truth).sum().item()
    union = (positive | truth).sum().item()
    dice_den = positive.sum().item() + truth.sum().item()
    return intersection, union, dice_den

def save_checkpoint(next_epoch, best_val_iou, history, global_step, name='latest.pt'):
    payload = {
        'next_epoch': next_epoch, 'best_val_iou': best_val_iou,
        'history': history, 'global_step': global_step,
        'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
        'config': {'image_size': IMAGE_SIZE, 'architecture': ARCHITECTURE,
                    'num_classes': 2, 'seed': SEED},
    }
    temporary = RUN_ROOT / f'.{name}.tmp'
    torch.save(payload, temporary)
    temporary.replace(RUN_ROOT / name)

latest = RUN_ROOT / 'latest.pt'
history, best_val_iou, global_step, start_epoch = [], -1.0, 0, 0
if latest.exists():
    state = torch.load(latest, map_location=device)
    model.load_state_dict(state['model'])
    optimizer.load_state_dict(state['optimizer'])
    scheduler.load_state_dict(state['scheduler'])
    scaler.load_state_dict(state['scaler'])
    history = state.get('history', [])
    best_val_iou = state.get('best_val_iou', -1.0)
    global_step = state.get('global_step', 0)
    start_epoch = state.get('next_epoch', 0)
    print(f'Resuming at epoch {start_epoch + 1}; best val IoU={best_val_iou:.4f}')
else:
    print('No checkpoint found; starting from ImageNet-pretrained backbone.')

## 3. Resumable training

    Stop this cell at any time and rerun it later. The periodic checkpoint restarts the current epoch if Colab disconnected mid-epoch; completed epochs are never repeated. The time limit is checked between batches so the cell exits cleanly before the free-session budget is exhausted.

In [ ]:
from tqdm.auto import tqdm
    
session_deadline = time.time() + MAX_MINUTES_PER_SESSION * 60
try:
    for epoch in range(start_epoch, MAX_EPOCHS):
        model.train()
        train_loss = 0.0
        progress = tqdm(loaders['train'], desc=f'Epoch {epoch + 1}/{MAX_EPOCHS}')
        for images, masks in progress:
            if time.time() >= session_deadline:
                save_checkpoint(epoch, best_val_iou, history, global_step)
                print('Session time limit reached; resume from latest.pt next time.')
                raise TimeoutError
            images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                output = model(images)
                loss = criterion(output['out'], masks) + 0.5 * dice_loss(output['out'], masks)
                if 'aux' in output:
                    loss = loss + 0.2 * criterion(output['aux'], masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            global_step += 1
            progress.set_postfix(loss=f'{loss.item():.3f}')
            if global_step % CHECKPOINT_EVERY_STEPS == 0:
                save_checkpoint(epoch, best_val_iou, history, global_step)

        model.eval()
        val_loss, val_intersection, val_union, val_dice_den = 0.0, 0, 0, 0
        with torch.no_grad():
            for images, masks in loaders['val']:
                images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                    output = model(images)['out']
                    val_loss += (criterion(output, masks) + dice_loss(output, masks)).item()
                a, b, c = scores(output, masks)
                val_intersection += a; val_union += b; val_dice_den += c
        val_iou = val_intersection / max(val_union, 1)
        val_dice = 2 * val_intersection / max(val_dice_den, 1)
        row = {'epoch': epoch + 1, 'train_loss': train_loss / max(len(loaders['train']), 1),
                'val_loss': val_loss / max(len(loaders['val']), 1), 'val_iou': val_iou, 'val_dice': val_dice}
        history.append(row)
        scheduler.step()
        save_checkpoint(epoch + 1, max(best_val_iou, val_iou), history, global_step)
        if val_iou > best_val_iou:
            best_val_iou = val_iou
            save_checkpoint(epoch + 1, best_val_iou, history, global_step, 'best.pt')
        print(row)
except TimeoutError:
    pass
except KeyboardInterrupt:
    save_checkpoint(start_epoch, best_val_iou, history, global_step)
    raise

print(f'Finished through epoch {history[-1]["epoch"] if history else start_epoch}.')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

if history:
    display(pd.DataFrame(history))
    pd.DataFrame(history).to_csv(RUN_ROOT / 'history.csv', index=False)
    pd.DataFrame(history).plot(x='epoch', y=['val_iou', 'val_dice'], marker='o', ylim=(0, 1))
    plt.grid(); plt.show()
else:
    print('No completed epoch yet. Rerun training to continue.')

## 4. Evaluate, export, and test one image

    `best.pt` is the resumable PyTorch checkpoint. `plantseg_lesion_segmentation.ts` is the deployable TorchScript file; it accepts a normalized `[1, 3, IMAGE_SIZE, IMAGE_SIZE]` tensor and returns two logits `[1, 2, IMAGE_SIZE, IMAGE_SIZE]`. Keep `model_config.json` beside it so the app knows preprocessing and class meaning.

In [ ]:
best = RUN_ROOT / 'best.pt'
if not best.exists():
    raise FileNotFoundError('best.pt does not exist yet; complete at least one validation epoch')
state = torch.load(best, map_location=device)
model.load_state_dict(state['model'])
model.eval()
intersection, union, dice_den = 0, 0, 0
with torch.no_grad():
    for images, masks in tqdm(loaders['test'], desc='test'):
        images, masks = images.to(device), masks.to(device)
        output = model(images)['out']
        a, b, c = scores(output, masks)
        intersection += a; union += b; dice_den += c
print({'test_iou': intersection / max(union, 1), 'test_dice': 2 * intersection / max(dice_den, 1)})

class LogitOnly(nn.Module):
    def __init__(self, segmentation_model):
        super().__init__(); self.segmentation_model = segmentation_model
    def forward(self, x):
        return self.segmentation_model(x)['out']

export_model = LogitOnly(model).eval().cpu()
example = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
traced = torch.jit.trace(export_model, example)
traced.save(str(RUN_ROOT / 'plantseg_lesion_segmentation.ts'))
(RUN_ROOT / 'model_config.json').write_text(json.dumps({
'architecture': ARCHITECTURE, 'task': 'binary lesion segmentation',
    'input_size': [IMAGE_SIZE, IMAGE_SIZE], 'normalization_mean': [0.485, 0.456, 0.406],
    'normalization_std': [0.229, 0.224, 0.225], 'output_channels': {'0': 'background', '1': 'diseased_region'},
    'source': 'PlantSeg, https://doi.org/10.5281/zenodo.17719108', 'license': 'CC BY-NC 4.0'
}, indent=2))
print('Exported:', RUN_ROOT / 'plantseg_lesion_segmentation.ts')

In [ ]:
import matplotlib.pyplot as plt
sample_image, sample_mask = next(iter(loaders['test']))
with torch.no_grad():
    prediction = model(sample_image.to(device))['out'].argmax(1).cpu()[0].numpy()
image = sample_image[0].permute(1, 2, 0).numpy()
image = np.clip(image * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]), 0, 1)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image); axes[0].set_title('image')
axes[1].imshow(sample_mask[0], cmap='gray'); axes[1].set_title('ground truth')
axes[2].imshow(image); axes[2].imshow(prediction, alpha=0.45, cmap='Reds'); axes[2].set_title('prediction')
for ax in axes: ax.axis('off')
plt.show()

# Optional: package only the model artifacts, not the licensed image archive.
export_zip = WORK_ROOT / 'plantseg_lesion_model.zip'
with zipfile.ZipFile(export_zip, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for name in ('plantseg_lesion_segmentation.ts', 'model_config.json', 'best.pt', 'history.csv'):
        path = RUN_ROOT / name
        if path.exists(): z.write(path, name)
print('Model package:', export_zip)

### Inference contract for the app

1. Load the TorchScript file with `torch.jit.load(...).eval()`.
2. Convert an RGB image to a tensor, resize to 256x256, divide by 255, then normalize with the mean/std in `model_config.json`.
3. Run the model and use `softmax(logits, dim=1)[:, 1]` as the diseased-region probability.
4. Resize the probability map back to the original image size and threshold it (start with 0.5, tune on validation data).

This model does not diagnose a disease name. Pair it with a separately validated classifier if the app needs disease labels, and do not treat either output as agronomic advice without expert review.